Explore the local launcher, the Right job monitor, and cooperative stop handling. The example simulates a Right failure locally.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.app.runner import launch, run_left, run_outcome, right_job_params
from pair_recon.app.runner import RightJobMonitor, StageWatchdog
from pair_recon.app.runner import record_runtime_failure, make_run_id

show(launch, run_left, run_outcome, right_job_params, record_runtime_failure, make_run_id)
show_methods(RightJobMonitor, 'start', 'request_cancel', 'failure', 'raise_if_failed', 'result')
show_methods(StageWatchdog, 'start', 'check', 'close')


## A Right failure triggers the local stop signal

A mock remote job fails immediately. The monitor runs in a local thread; no Databricks job is submitted.


In [ ]:
from unittest.mock import Mock
from pair_recon_left.common.runtime.stop import StopSignal, StopRequested

remote = Mock()
remote.run_id = 123
remote.result.side_effect = RuntimeError("toy Right failure")
stop = StopSignal()
monitor = RightJobMonitor(remote, stop=stop).start()
monitor.done.wait(timeout=5)
print("run id:", monitor.run_id)
print("monitor state:", monitor.final_state)
print("stop triggered:", stop.triggered)
try:
    stop.check()
except StopRequested as error:
    print("Left sees:", type(error).__name__, str(error))


## Right job parameters and outcome

Inputs are ordinary dictionaries. A mock Jobs API supplies the notebook result as JSON.


In [ ]:
from pprint import pprint
from types import SimpleNamespace
from unittest.mock import Mock
from datetime import datetime, timezone

settings = {"volume_root": "/Volumes/demo/recon/runs", "owner": "alex", "timezone": "UTC", "pair_table": "demo.recon.pair_recon_pair_meta", "status_table": "demo.recon.pair_recon_run_status", "error_table": "demo.recon.pair_recon_run_error", "right_workers": 2}
run_id = make_run_id("orders", "alex", "UTC", now=datetime(2026, 1, 2, tzinfo=timezone.utc))
print("run id input:", "orders", "alex", "UTC", "2026-01-02")
print("run id output:", run_id)
print("settings:"); pprint(settings)
print("job parameters:"); pprint(right_job_params(settings, "orders", run_id, {"sample": 0.1}))
jobs = Mock()
jobs.get_run_output.return_value = SimpleNamespace(notebook_output=SimpleNamespace(result='{"state":"matched","count":3}'))
run = SimpleNamespace(run_id=42, tasks=[])
print("notebook output input:", jobs.get_run_output.return_value.notebook_output.result)
print("run outcome:", run_outcome(jobs, run))
print("SDK call:", jobs.get_run_output.call_args)
store = Mock()
record_runtime_failure(store, "orders", "right-job", RuntimeError("toy failure"))
print("failure artifact input:", store.put_json.call_args)


## Monitor cancellation and error propagation

These objects are local. `request_cancel` marks a caller-initiated cancellation; `failure`, `raise_if_failed`, and `result` expose a detected failure.


In [ ]:
from unittest.mock import Mock
remote = Mock()
remote.run_id = 123
monitor = RightJobMonitor(remote)
print("failure before error:", monitor.failure())
monitor.request_cancel()
print("cancel requested:", monitor.cancel_requested.is_set())
monitor.error = RuntimeError("toy failure")
monitor.done.set()
print("failure:", monitor.failure())
for method in (monitor.raise_if_failed, monitor.result):
    try:
        method()
    except RuntimeError as error:
        print(method.__name__, "raised:", str(error))


## `StageWatchdog.check`, `start`, and `close`

The mock store returns no state; timeouts are disabled for a deterministic no-stall example. The daemon is closed immediately.


In [ ]:
from unittest.mock import Mock
from pair_recon_left.common.runtime.stop import StopSignal
store = Mock()
store.exists.return_value = False
watchdog = StageWatchdog(store, StopSignal(), interval=60)
watchdog.fetch_stall = 0
watchdog.first_result = 0
watchdog.state_stall = 0
watchdog.volume_failures = 0
print("check output:", watchdog.check(now=100))
watchdog.start()
print("thread alive:", watchdog.thread.is_alive())
watchdog.close()
watchdog.thread.join(timeout=1)
print("thread stopped:", not watchdog.thread.is_alive())
